# Conclusion

**The question.** Using only information available *before an aircraft departs* — schedule, carrier, route — predict its arrival delay in minutes. A supervised regression judged primarily by **MAE**, on 2024 US domestic flights (~6.97 M after cleaning), under a strict **leakage boundary** (no departure / arrival actuals) and a **time-ordered** 70 / 10 / 20 split.

This notebook draws together what the EDA, feature engineering, and modelling produced.

## The model ladder (validation)

| model | MAE | RMSE | $R^2$ |
|---|---|---|---|
| mean baseline | 23.52 | 45.78 | ≈ 0 |
| **median baseline** | **18.54** | 45.67 | ≈ 0 |
| (carrier, dep_hour) mean | 23.10 | 45.68 | ≈ 0 |
| OLS / Ridge | 19.76 | 45.28 | 0.001 |
| LightGBM (L1) | 17.97 | 45.88 | ≈ 0 |
| **LightGBM (tuned, L1)** | **17.81** | 45.73 | ≈ 0 |

Two structural facts fall out at once:

- **The median is the floor to beat, not the mean.** On a right-skewed target the MAE-optimal constant is the *median* (18.54); squared-loss models (OLS / Ridge) predict the *mean*, so they score *worse* on MAE (19.76) even while capturing real signal.
- **Only the median-targeting L1 trees edge below the floor.** Tuned LightGBM reaches **17.81** — a genuine but small **0.73-minute** gain over the median. Regularisation and 25 trials of Optuna barely move it: with millions of rows and a handful of features ($n \gg p$) there is nothing to overfit, and $R^2$ stays at ≈ 0.

## The held-out test (spent once)

Each tuned model, selected on validation, is loaded from MLflow and scored a **single time** on the untouched final quarter (Oct–Dec 2024):

| model | test MAE | test RMSE | test $R^2$ |
|---|---|---|---|
| Naive (median) | 20.87 | 51.44 | −0.02 |
| Ridge | 22.52 | 50.75 | 0.01 |
| **Before-departure (tuned LightGBM)** | **20.36** | 51.72 | −0.03 |
| After-departure (diagnostic) | 7.32 | 11.02 | 0.953 |

- **The deployable answer is MAE ≈ 20.4 min** on genuinely unseen, out-of-season data — edging the naive median (20.87) by **0.50 min** (paired 95% CI **[0.494, 0.513]**, excludes zero), so the small schedule signal is **statistically real and generalises** — though tiny.
- **The val → test rise is distribution shift, not overfitting.** Every model — *including the parameter-free naive median* — worsens by ~2 min, because the test window is the harder holiday / winter season, not because the models overfit validation.
- **MAE vs RMSE, made concrete.** Ridge posts the best RMSE (it targets the mean) but the worst MAE; the L1 tree is the reverse (best MAE, higher RMSE). We optimise MAE, so the tree is the right choice — a genuine tradeoff, not a free lunch.

## How much is departure-time information worth?

*Why* does before-departure prediction plateau? An **after-departure control** answers it: the *same* model, split, and tuning, given the post-departure state (`dep_delay`, `taxi_out`, `wheels_off`).

| setting | test MAE | test $R^2$ |
|---|---|---|
| Before departure (schedule only) | 20.36 | −0.03 |
| After departure (+ departure state) | **7.32** | **0.953** |

Arrival delay is nearly **locked in at pushback**: $\text{arr\_delay} = \text{dep\_delay} + \Delta_{\text{enroute}}$, and the enroute term is small, so once departure is observed the identical model explains **95%** of the variance. This is a **benchmark of the value of departure-time information** — a ≈ 13-minute MAE gap (a 2.8× error) — and a confirmation that the pipeline extracts signal *when signal is present*.

It is **not** a proof that the schedule holds no signal — a high after-departure score cannot establish that negative. The before-departure ceiling is instead **mechanistic**: the drivers of delay — weather, ATC ground stops, cascading late aircraft (the BTS cause codes) — are *day-of realized events*, absent from the published schedule by construction.

## Two findings from the exploration

- **Punctuality ≠ reliability.** A Brown–Forsythe test rejects equal delay *spread* across carriers ($W = 3{,}526$, $p \approx 0$), but the effect size is tiny ($\eta^2 \approx 0.007$). Carriers differ in *consistency* (widest-to-narrowest IQR ratio ≈ 2.57×) on an axis **distinct from** their typical delay — the least-punctual carrier by median can be the most reliable by spread.
- **Significance ≠ importance.** At ~7 M rows every test is "significant" ($p \approx 0$), so the honest signal lives in **effect sizes** — and they stay small for every schedule feature. That is the same story the near-zero $R^2$ tells from the modelling side.

## What the result is good for

A low $R^2$ is the *finding*, not a failure — the project delivers a **quantified, honest map of what is knowable, and when**:

- **A pre-departure forecast cannot be squeezed from the schedule alone.** To be useful it must incorporate **external, day-of data** — weather forecasts, real-time NAS / ATC status, historical airport-hour congestion. That is a concrete design conclusion, not a dead end.
- **The value of information is quantified.** A delay estimate should update sharply the moment a flight departs (MAE ≈ 20 → 7); a passenger app or an operations desk should act on exactly that.
- **The systematic component is still useful.** Even at $R^2 \approx 0$ the model captures the *conditional average* — which carriers, airports, and hours run late — which supports planning ("add a buffer for the evening ORD bank") even though the day-specific minute is dominated by irreducible noise.

## Limitations & future work

- **External day-of data.** The clearest path to a genuinely useful before-departure forecast is adding weather forecasts and real-time NAS / ATC status — the causal drivers the schedule lacks.
- **Probabilistic output.** Given the irreducible variance, a **quantile / distributional** forecast ("25% chance of > 30 min late") is more honest and actionable than a single-number point estimate.
- **Aircraft rotation.** Delay propagation is one of the strongest real predictors, but this dataset has no tail number, so a plane's earlier-legs state can't be reconstructed; a dataset with `TAIL_NUM` would enable it — handled carefully, since it is itself a day-of signal.

---

**Bottom line.** Before departure, a US domestic flight's arrival delay is predictable to about **±20 minutes** — barely better than the historical median — because the *causes* of delay have not yet happened. The moment the flight leaves, the same model is accurate to about **±7 minutes**. The project's contribution is to establish that boundary rigorously, under a clean leakage discipline, and to quantify exactly what crossing it is worth.

## Self-assessment

Rather than assign ourselves a numeric score, a candid reflection on **what went well** and **what could be improved** — which, by the project's own logic, is part of the analysis: estimating the achievable ceiling, reaching it, and locating the remaining headroom.

**What went well**

- **The ceiling was estimated, reached, and bounded.** The after-departure control quantifies the value of the missing day-of information ($\approx 13$ min of MAE), and a paired bootstrap CI shows the deployable model *provably* — if narrowly — beats the naive baseline. "Achieved $\approx$ achievable."
- **Applied, not decorative, mathematics.** MAE / median optimality, gradient boosting, Optuna's TPE, Brown–Forsythe with $\eta^2 / \omega^2$, cyclical encoding, and the $\text{arr\_delay}=\text{dep\_delay}+\Delta_{\text{enroute}}$ identity — each tied to a concrete modelling decision.
- **Reproducible and checked.** A pinned environment, a `pytest` suite for the `src/` helpers, seeded tuning, and MLflow run tracking.

**What could be improved**

- **External, day-of data is the single biggest lever** — weather forecasts, real-time ATC / NAS status — which this dataset does not contain, and the clearest path to a genuinely useful before-departure forecast.
- **A probabilistic / quantile output** (e.g. $P(\text{delay} > 15\text{ min})$, or a P90 delay) would be more honest and actionable than a point estimate, given the irreducible day-of variance.
- **Broader validation** — a blocked time-series cross-validation would turn the single-split point estimates into distributions (though at $n \approx 1.4$M they are already precise).
- **More of the pipeline could live in `src/`** — the notebooks carry cleaning and feature logic that isn't unit-tested; extracting and testing it would raise robustness.
- **A fuller survey of comparable published approaches** would strengthen the framing beyond the brief related-work note.

The list mirrors the project's arc: we know roughly what the schedule *can* achieve, we reached it, and every improvement above is about the **information or engineering** needed to go further — not about squeezing the current data harder.